# 02 · Step-by-step exploration: NER, then targeted tone

**Scope.** German only, one block-A vote (**A-631**, Begrenzungsinitiative,
2020-09-27), every person and organisation, tone *toward* the actor only.
Each step's output is read before the next one starts.

The computation lives in the `mediapos` package (`preprocess`, `relevance`,
`ner`, `actors`, `linking`); this notebook runs it step by step and shows the
results.

## 1. The slice

The archive for A-631 holds every article from the 34 outlets published inside
the event window. The query filtered on outlet, dates and language, **not on
topic**. This step keeps the German rows and checks them against the pull
receipt. Nothing here prints article text, so the notebook can be committed
with its outputs.

In [1]:
import asyncio

import anthropic
import pandas as pd
from dotenv import load_dotenv

from mediapos import config as cfg
from mediapos.actors import actor_tables, actors_from_mentions
from mediapos.linking import (
    LINK_CACHE,
    WIKIDATA_CACHE,
    ModelCache,
    WikidataCache,
    entity_tables,
    is_vague,
    link_actors,
    resolve_vague,
)
from mediapos.ner import tag_all, tag_report
from mediapos.preprocess import (
    add_text,
    check_text,
    event_archive,
    keep_longest,
    load_slice,
    read_random,
    select_vote,
)
from mediapos.relevance import (
    CONCURRENCY,
    PROMPT_SHA,
    add_labels,
    cost,
    kept_articles,
    label_missing,
    load_labels,
    load_review,
)

load_dotenv(cfg.REPO_ROOT / ".env")

EVENT_ID = "A-631"
LANGUAGE = "de"

query, receipt, archive = event_archive(EVENT_ID)
print(f"event    {EVENT_ID}  window {query.date_from} → {query.date_to}")
print(f"archive  {archive.relative_to(cfg.REPO_ROOT)}")
print(f"receipt  {receipt['articles_read']:,} articles, {receipt['by_language']}")

event    A-631  window 2020-08-28 → 2020-10-11
archive  data/corpus/2020-08-28_2020-10-11.tsv.xz
receipt  87,403 articles, {'de': 58649, 'fr': 28754}


In [2]:
articles = load_slice(archive, LANGUAGE)

# The rows must be exactly what the receipt says was downloaded.
assert len(articles) == receipt["by_language"][LANGUAGE], len(articles)

print(f"{len(articles):,} {LANGUAGE} rows, count matches the receipt")
print(f"published {articles.pubtime.min():%Y-%m-%d} → {articles.pubtime.max():%Y-%m-%d}")

58,649 de rows, count matches the receipt
published 2020-08-27 → 2020-10-11


### One id, several rows: revisions of the same article

Some ids appear two or three times. These rows have the same outlet and
timestamp but a different `content_id` and body: they are **edited versions**
of one online article. Counting every version would weight an article by how
often it was edited, so we keep **one row per id: the longest version**, with
ties broken by `content_id` so the choice is deterministic.

In [3]:
rows = len(articles)
articles = keep_longest(articles)
assert articles["id"].is_unique
print(f"{rows - len(articles)} extra versions removed; slice: {len(articles):,} articles, one per id")

62 extra versions removed; slice: 58,587 articles, one per id


In [4]:
(articles.groupby(["medium_code", "medium_name"]).size()
         .rename("articles").sort_values(ascending=False).reset_index())

,medium_code,medium_name,articles
0,AZM,Aargauer Zeitung / MLZ,4567
1,SGTO,tagblatt.ch,4392
2,BLIO,blick.ch,4239
3,NNBE,bernerzeitung.ch,4007
4,SRF,srf.ch,3601
5,NNBS,bazonline.ch,3175
6,NNTA,tagesanzeiger.ch,3149
7,ZWAO,20 minuten online,3145
8,NNBU,derbund.ch,3053
9,LUZ,Luzerner Zeitung,2881


## 2. The articles about the vote

The window holds everything the outlets published, so we keep the articles that
**name the initiative**: its common name (*Begrenzungsinitiative*), its
opponents' name (*Kündigungsinitiative*) or its official title (*massvolle
Zuwanderung*). We search the headline, subhead and body.

*Personenfreizügigkeit* is left out on purpose. It also appears in articles
about the EU framework agreement, so it would pull in more than this vote.

Naming the initiative once does not make an article *about* it. A portrait of
the cantonal CVP can mention it in passing. So we also count the mentions and
record whether the name is in the headline, which shows how central the vote
is to each article.

In [5]:
vote = select_vote(articles)

print(f"{len(vote):,} of {len(articles):,} articles name the initiative "
      f"({100 * len(vote) / len(articles):.1f}%)")
print(f"{vote['vote_in_head'].sum()} name it in the headline or subhead")
(vote.groupby("medium_code")
     .agg(articles=("id", "size"), in_head=("vote_in_head", "sum"),
          median_mentions=("n_vote_mentions", "median"))
     .join(articles.groupby("medium_code").size().rename("all_articles"))
     .assign(share_pct=lambda t: (100 * t.articles / t.all_articles).round(1))
     .sort_values("articles", ascending=False))

1,086 of 58,587 articles name the initiative (1.9%)
82 name it in the headline or subhead


,articles,in_head,median_mentions,all_articles,share_pct
medium_code,,,,,
SGTO,218,9,3.0,4392,5.0
AZM,79,0,1.0,4567,1.7
SRF,64,12,3.0,3601,1.8
NNBE,60,12,1.5,4007,1.5
NNBS,57,12,2.0,3175,1.8
SGT,53,0,1.0,2794,1.9
NNBU,53,12,2.0,3053,1.7
NNTA,50,11,2.0,3149,1.6
LUZ,48,1,1.0,2881,1.7


## 3. From markup to text

`content` is not plain text. It is XML-like markup, and each tag needs a decision:

| Tag | What it is | Decision |
|---|---|---|
| `<ld>` | lead paragraph | **body** |
| `<p>` | paragraph | **body** |
| `<zt>` | crosshead (subheading) | **body**, on its own line |
| `<a>`, `<br>`, tables | links, line breaks, cells | **body**, the markup removed and the text kept |
| `<lg>` | photo caption | own column `captions` |
| `<ka>` | box, often a pull quote that repeats the body | own column `boxes` |
| `<nt>` | note, e.g. a line about the author | own column `notes` |
| `<au>` | author byline | **dropped** |
| `<ur>` | photo credit | **dropped** |

Bylines and photo credits are dropped because NER would read the journalist's
or photographer's name as an actor in the story. Captions, boxes and notes are
kept apart rather than thrown away. Captions are visibility too (a photo of an
actor), and pull quotes would count the same mention twice if merged into the
body. Whether to use them is a later decision.

The model input `text` is headline, subhead and body, separated by blank
lines.

In [6]:
vote = add_text(vote)
words = check_text(vote)  # no tag or HTML entity left, every word accounted for

print("no tag or HTML entity left; every word accounted for")
print("words:", {k: f"{v:,}" for k, v in words.items()})

no tag or HTML entity left; every word accounted for
words: {'body': '2,829,365', 'captions': '260,528', 'boxes': '55,690', 'notes': '543', 'dropped': '2,170'}


## 4. Is the vote the main subject?

Naming the initiative is not the same as being about it. An article on the
hunting-law vote can name the initiative three times ("the same Sunday we also
vote on…"). So a model reads each article and gives **one label**:

- **related**: the main subject is the Begrenzungsinitiative or its direct
  stakes (the campaign, its actors and arguments, the result, free movement with
  the EU and the bilateral agreements as they bear on this vote).
- **not_related**: anything else, including the other four votes of
  27 September and **roundups covering several votes side by side**. In a
  roundup we cannot tell which mentions belong to the initiative.

**Model:** Claude Haiku 5.5, thinking off, low effort, with the output
constrained to the two labels. **Input:** headline, subhead and the first
4,000 characters of the body.

**Cache:** each label is saved to `data/interim/a631_relevance.jsonl` under the
article id, the model and a hash of the prompt. A rerun only calls the model for
articles with no saved label. Changing the prompt, the model or the truncation
changes the hash, so everything is labelled again. No article text is saved.

**This filter is a model and has an error rate.** It sits upstream of every
later step, so its labels are reviewed by hand below.

In [7]:
already = len(load_labels())
failures = await label_missing(vote)
labels = load_labels()
vote = add_labels(vote)

print(f"prompt {PROMPT_SHA}: {already} labels cached, {len(labels) - already} new, {len(failures)} failed")
for failure in failures:
    print("  ", failure)
print(cost(list(labels.values())))
(vote.groupby("medium_code")
     .agg(named=("id", "size"), related=("relevance", lambda s: (s == "related").sum()))
     .assign(kept_pct=lambda t: (100 * t.related / t.named).round(0))
     .sort_values("named", ascending=False))

prompt 315452286fb2: 1086 labels cached, 0 new, 0 failed
2,421,585 tokens in, 11,000 out ≈ $0.25


,named,related,kept_pct
medium_code,,,
SGTO,218,20,9.0
AZM,79,32,41.0
SRF,64,33,52.0
NNBE,60,29,48.0
NNBS,57,30,53.0
SGT,53,30,57.0
NNBU,53,28,53.0
NNTA,50,27,54.0
LUZ,48,28,58.0


### Hand review of the labels

Haiku's label left off-topic articles in: it reads only the start of each article,
and the two largest "related" articles turned out to be local city tickers. The
251 articles with a doubtful headline were reviewed from their lead and the
passages about the vote, and ranked by how the vote features. The other 222 were
kept on their headline alone. This is one reviewer's judgement, not a measured
accuracy.

| Tier | The vote is | Action |
|---|---|---|
| `A_main`, `A_headline_only` | the main topic | keep |
| `B_important` | one of the important topics | keep |
| `C_side` | a side topic the article engages with | keep |
| `D_framework` | only the trigger of an article on the EU framework agreement | remove |
| `D_roundup` | one of several proposals covered side by side | remove |
| `D_passing` | mentioned in passing | remove |
| `E_page` | a small part of a ticker, a news brief or a mixed page of letters | remove |


In [8]:
review = load_review()
related = kept_articles(vote, review)
texts = dict(zip(related["id"], related["text"]))

print(f"{len(review)} labelled related, {len(review) - len(related)} removed by hand, {len(related)} kept")
review.groupby(["action", "tier"]).agg(articles=("id", "size"), stories=("story", "nunique"))

473 labelled related, 130 removed by hand, 343 kept


articles  stories
action tier                              
keep   A_headline_only       222        1
       A_main                 46       30
       B_important            55       33
       C_side                 20       10
remove D_framework            50       26
       D_passing              46       24
       D_roundup              12        9
       E_page                 22       19

## 5. Named entities

Haiku tags the **343 articles left after the hand review**. The output is **one
row per mention**: article, character position, mention text, type and entity
id. The positions are kept because the tone step needs to know where each
mention stands. spaCy and GLiNER were compared with Haiku on the same articles:
see `03_compare_ner_de.ipynb`.

- **Haiku marks every mention inside the text** and gives it an entity id:
  *‹P1›Tiana Angelina Moser‹/P1› … ‹P1›Moser‹/P1› … die ‹O1›FDP‹/O1›-Chefin*. It
  works through the article in pieces of at most 400 words and gets the table
  of entities found so far with each piece, so *Moser* in a later piece keeps
  id P1. The ids link spellings **within** an article; linking across articles
  comes in the normalization step. A first version asked Haiku for a list of
  names and searched for them; it lost every spelling missing from the list
  (mostly surnames standing alone) and was replaced.
- **Haiku has to copy the text to tag it, and it does not always copy
  faithfully**: it corrects a case ending, adds a word, chats ("wait, I must
  copy exactly…"), and above all **skips headlines and sometimes whole
  passages**. The copy is aligned with the original and each tag carried back to
  its original position; a tag is kept only if its text stands unchanged in the
  original. Every passage the copy skipped is sent again on its own, with the
  entity table, and tagged in a second pass (a third for what that one skips).
  What remains untagged after that is counted and reported.
- **Haiku sometimes leaves a repeat untagged**: it tags *Bundesrat* in one
  sentence and not in the next. Every untagged occurrence of a form Haiku tagged
  elsewhere in the same article is filled in with that form's id (longest form
  first, whole words only). These mentions are marked `filled`, so they can
  always be counted apart or removed. A form Haiku tagged wrongly once is
  filled in wrongly too; that is the price of the fill.

Haiku's tags are saved in `data/interim/` under a hash of the prompt and
settings, so a rerun reads them.


In [9]:
haiku_found = await tag_all(related)

# Every stored position must point at the stored mention text.
assert all(texts[r.id][r.start:r.end] == r.mention for r in haiku_found.itertuples()), "a position is off"
tag_report(related, haiku_found)

Haiku: 9,234 mentions of 3,292 entities
copy repairs: 15,343 skipped characters tagged in a second pass, 36 left untagged, 298 tags dropped
filled in: 541 untagged repeats of a tagged form (5.9% of Haiku's mentions, column `filled`)
1,413,584 tokens in, 1,242,255 out ≈ $0.76


## 6. Normalization: from mentions to actors

Haiku's ids link the spellings of an entity **within** an article. This step
links them **across** articles, so that each actor gets one key, with its
spellings (variants), a count per article, and a total. The next step,
Wikidata linking, is also where *SVP* and
*Schweizerische Volkspartei* will merge through Wikidata's aliases.

| Step | Example | Rule |
|---|---|---|
| Spelling key | *Operation-Libero* = *Operation Libero*, *Cassis’* = *Cassis* | case, hyphens, spacing and outer quotes ignored |
| Case endings | *Blochers* → *Blocher* | *-s/-es* removed when the stem is also a spelling |
| Junk | *und*, *614*, *Begrenzungsinitiative* | function words, no letters, ballot proposals and laws |
| Titles | *der Bundespräsident Ueli Maurer* → *Ueli Maurer* | leading titles and articles removed from person names; a title alone tagged as a person is dropped |
| Misfiled spellings | *Marco Chiesa* under Cassis's id | a person spelling whose words are not all in the entity's name becomes an entity of its own |

**Persons** are keyed by full name. A name that is the word-by-word ending of
exactly **one** longer name is attached to it (*Cassis* → *Ignazio Cassis*). A
surname with several candidates (*Sommaruga*: Simonetta or Carlo) stays on its
own with status `partial`; guessing would merge two people.

**Organizations** are keyed by vote: in each article entity the most common
spelling is the head, and each spelling goes to the head it was grouped with
most often across articles. A one-off mix-up in one article is outvoted.

A person and an organization never share a key (`P:` / `O:`).

In [10]:
actor_mentions = actors_from_mentions(haiku_found)
variants, per_article, actors = actor_tables(actor_mentions, related)

share = actors["mentions"].cumsum() / actors["mentions"].sum()
print(f"{len(actor_mentions):,} mentions -> {len(actors):,} actors "
      f"({(actors.type == 'person').sum()} persons, {(actors.type == 'organization').sum()} organizations); "
      f"{len(haiku_found) - len(actor_mentions)} mentions dropped as junk")
print(f"misfiled person spellings split off: {actor_mentions['entity'].str.contains('~').sum()}")
print(f"top 50 actors carry {100 * share.iloc[49]:.0f}% of mentions, top 200 {100 * share.iloc[199]:.0f}%; "
      f"{(actors.articles == 1).sum()} actors appear in one article only")
print("person mentions by status:",
      actor_mentions[actor_mentions.actor.str[0] == "P"]["status"].value_counts().to_dict())

9,139 mentions -> 679 actors (393 persons, 286 organizations); 95 mentions dropped as junk
misfiled person spellings split off: 15
top 50 actors carry 74% of mentions, top 200 91%; 343 actors appear in one article only
person mentions by status: {'full': 2992, 'attached': 108, 'partial': 64}


In [11]:
spellings = {a: ", ".join(f"{m} ({n})" for m, n in zip(g["mention"], g["mentions"]))
             for a, g in variants.groupby("actor")}
top = actors.head(30).assign(spellings=lambda t: t["actor"].map(spellings))
top[["name", "type", "mentions", "articles", "spellings"]]

,name,type,mentions,articles,spellings
0,SVP,organization,1801,300,"SVP (1779), Volkspartei (14), Schweizerische V..."
1,EU,organization,1463,218,"EU (1428), Europäischen Union (27), Europäisch..."
2,Bundesrat,organization,485,120,"Bundesrat (440), Bundesrats (32), Bundesrätin ..."
3,Karin Keller-Sutter,person,285,72,"Keller-Sutter (160), Karin Keller-Sutter (119)..."
4,Operation Libero,organization,262,17,"Operation Libero (191), Operation (40), Libero..."
5,Laura Zimmermann,person,150,14,"Zimmermann (77), Laura Zimmermann (69), Zimmer..."
6,FDP,organization,140,84,"FDP (133), Freisinnige (4), Freisinn (3)"
7,Christoph Blocher,person,134,35,"Blocher (68), Christoph Blocher (52), Blochers..."
8,Pierre-Yves Maillard,person,101,19,"Maillard (67), Pierre-Yves Maillard (33), Mail..."
9,Daniel Maerki,person,101,9,"Maerki (69), Daniel Maerki (24), Maerkis (8)"


## 7. Wikidata linking

Each actor gets a Wikidata QID, so that every spelling of an entity (*SVP*,
*Schweizerische Volkspartei*) ends up under one id, and its mentions can be
summed per article and across articles.

**Vague names first, article by article.** A lone surname (*Sommaruga*) or a
generic organization (*Parlament*, *Kantonspolizei*) means a different entity in
different articles. Haiku reads the sentences of one article and names the
entity meant (*Parlament* → *Bundesversammlung*), or answers *unknown*. It can
also correct the type: *SVP* tagged as a person is the party.

**Then one decision per actor.** Wikidata is searched under the actor's key, its
most frequent spelling and its longest one, and the candidates are merged.

| Case | Decision |
|---|---|
| exactly one Swiss candidate of the right kind (human for a person) | `rule`, no model |
| several, or only foreign ones | Haiku picks one from sentences across articles, or `none` |
| nothing fits | Haiku gives the official name (*SP* → *Sozialdemokratische Partei der Schweiz*), it is searched, and the decision is made again (`renamed, …`) |

Generic organizations left unresolved stay unlinked (`vague`). Wikidata answers
and Haiku answers are cached under `data/interim/`; requests to Wikidata go one
per second, and an error answer (*search too busy*) is retried, never cached.


In [12]:
wd = WikidataCache(WIKIDATA_CACHE)
link_cache = ModelCache(LINK_CACHE)
client, gate = anthropic.AsyncAnthropic(), asyncio.Semaphore(CONCURRENCY)

resolved = await resolve_vague(actor_mentions, texts, link_cache, client, gate)
links = await link_actors(resolved, texts, wd, link_cache, client, gate)
links["mentions"] = links["actor"].map(resolved["actor"].value_counts())

print(f"vague mentions re-keyed in their article: "
      f"{resolved['resolved_from'].notna().sum()} of {is_vague(actor_mentions).sum()}")
(links.groupby("method").agg(actors=("actor", "size"), mentions=("mentions", "sum"))
 .assign(share=lambda t: (100 * t["mentions"] / t["mentions"].sum()).round(1))
 .sort_values("mentions", ascending=False))

vague mentions re-keyed in their article: 112 of 130


,actors,mentions,share
method,,,
model,134,4429,48.5
rule,266,3509,38.4
"renamed, model: none",41,327,3.6
"renamed, rule",43,305,3.3
"renamed, none",112,302,3.3
"renamed, model",19,145,1.6
model: none,15,62,0.7
none,29,31,0.3
place,10,25,0.3


### One entity per QID

`entity_per_article` is the per-article count; summing it over articles gives each entity's total.

In [13]:
linked, entity_per_article, entities = entity_tables(resolved, links, related)

merged = entities[entities["actors"].str.len() > 1]
print(f"{len(links):,} actors -> {len(entities):,} entities; {len(merged)} entities merge several actors")
print(f"mentions under a QID: {100 * linked['qid'].notna().mean():.1f}%")
merged.head(15)[["label", "qid", "mentions", "actors"]]

673 actors -> 621 entities; 44 entities merge several actors
mentions under a QID: 92.1%


,label,qid,mentions,actors
0,Schweizerische Volkspartei,Q385258,1803,"[O:schweizerische volkspartei, O:svp]"
1,Europäische Union,Q458,1477,"[O:brüssel, O:eu, O:europäische union]"
2,Schweizerischer Bundesrat,Q30917,501,"[O:bundesbern, O:bundesrat, O:landesregierung,..."
3,Karin Keller-Sutter,Q115881,286,"[P:karin keller sutter, P:keller suter]"
7,Christoph Blocher,Q123857,139,"[P:blocher, P:christoph blocher]"
11,Grüne Schweiz,Q659739,85,"[O:grüne, O:grünen]"
13,Sozialdemokratische Partei der Schweiz,Q303745,78,"[O:sp, O:sp schweiz]"
15,Christlichdemokratische Volkspartei,Q659461,72,"[O:cvp, O:cvp schweiz]"
18,Bundesversammlung,Q684719,67,"[O:bundesversammlung, O:schweizer parlament]"
20,Ueli Maurer,Q123979,56,"[P:ueli mauer, P:ueli maurer]"


In [14]:
# The A-631 reference entities (party recommendations, rapporteurs): found, and how often?
reference = (pd.read_parquet(cfg.DATA_OUT / "entities.parquet")
             .query("event_id == @EVENT_ID and lang == 'de'")[["qid", "label", "role"]].drop_duplicates("qid"))
reference.merge(entities[["qid", "mentions", "articles", "actors"]], on="qid", how="left").fillna({"mentions": 0})

,qid,label,role,mentions,articles,actors
0,Q121470,Kurt Fluri,rapporteur,6.0,2.0,[P:kurt fluri]
1,Q1160790,Daniel Fässler,rapporteur,0.0,NaN,NaN
2,Q18413490,Valérie Piller Carrard,rapporteur,0.0,NaN,NaN
3,Q151768,Bürgerlich-Demokratische Partei,recommends_no,8.0,8.0,[O:bdp]
4,Q659461,Christlichdemokratische Volkspartei,recommends_no,72.0,46.0,"[O:cvp, O:cvp schweiz]"
5,Q667836,Eidgenössisch-Demokratische Union,recommends_yes,1.0,1.0,[O:edu]
6,Q667718,Evangelische Volkspartei,recommends_no,3.0,3.0,[O:evp]
7,Q202638,FDP.Die Liberalen,recommends_no,140.0,84.0,[O:fdp]
8,Q545900,Grünliberale Partei,recommends_no,21.0,17.0,"[O:glp, O:grünliberalen]"
9,Q659739,Grüne Partei der Schweiz,recommends_no,85.0,20.0,"[O:grüne, O:grünen]"


## Read a random article

`read_random(vote)` draws one article from the vote slice and prints it in full.
Narrow the draw with `relevance="related"` / `"not_related"` and/or an outlet
code, e.g. `read_random(vote, "not_related", outlet="SGTO")`.

The call below is commented out so that a full run prints no article.
**Clear this cell's output before committing.** Article text is licensed corpus
content and must not enter git.


In [37]:
read_random(vote, "related")

ValueError: Operands are not aligned. Do `left, right = left.align(right, axis=1)` before operating.